# Validation Document Length Distribution

This notebook scans the SentencePiece validation shards, treats `BOS_ID=1` as the document delimiter, and summarizes the raw token-length distribution of validation documents.

Outputs:
- total validation tokens and total documents
- summary stats for document length in tokens
- share of docs that exceed common context lengths
- histogram and CCDF plots

In [1]:
from pathlib import Path
import glob
import os
import numpy as np
import matplotlib.pyplot as plt

def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find repo root containing pyproject.toml")

REPO_ROOT = find_repo_root()
DATA_PATH = REPO_ROOT / "data/datasets/fineweb10B_sp1024"
VAL_GLOB = str(DATA_PATH / "fineweb_val_*.bin")
BOS_ID = 1

def load_data_shard(path: Path) -> np.ndarray:
    header = np.fromfile(path, dtype="<i4", count=256)
    if header.size != 256 or int(header[0]) != 20240520 or int(header[1]) != 1:
        raise ValueError(f"Unexpected shard header for {path}")
    num_tokens = int(header[2])
    header_bytes = 256 * np.dtype("<i4").itemsize
    toks = np.fromfile(path, dtype="<u2", count=num_tokens, offset=header_bytes)
    if toks.size != num_tokens:
        raise ValueError(f"Short read for {path}")
    return toks

val_files = [Path(p) for p in sorted(glob.glob(VAL_GLOB))]
if not val_files:
    raise FileNotFoundError(f"No validation shards found for {VAL_GLOB}; cwd={Path.cwd()} repo_root={REPO_ROOT}")

val_files

[PosixPath('/Users/robertgordan/Projects/parameter-golf/data/datasets/fineweb10B_sp1024/fineweb_val_000000.bin')]

In [2]:
def compute_doc_lengths(files, bos_id=BOS_ID):
    lengths = []
    total_tokens = 0
    prev_bos = None

    for file in files:
        toks = load_data_shard(file)
        bos_positions = np.flatnonzero(toks == bos_id)
        for pos in bos_positions:
            global_pos = total_tokens + int(pos)
            if prev_bos is not None:
                doc_len = global_pos - prev_bos
                if doc_len > 0:
                    lengths.append(doc_len)
            prev_bos = global_pos
        total_tokens += int(toks.size)

    if prev_bos is None:
        return np.array([total_tokens], dtype=np.int64), total_tokens

    final_len = total_tokens - prev_bos
    if final_len > 0:
        lengths.append(final_len)

    return np.asarray(lengths, dtype=np.int64), total_tokens

doc_lengths, total_tokens = compute_doc_lengths(val_files)

summary = {
    "num_docs": int(doc_lengths.size),
    "total_tokens": int(total_tokens),
    "mean": float(doc_lengths.mean()),
    "median": float(np.median(doc_lengths)),
    "min": int(doc_lengths.min()),
    "max": int(doc_lengths.max()),
    "p90": float(np.percentile(doc_lengths, 90)),
    "p95": float(np.percentile(doc_lengths, 95)),
    "p99": float(np.percentile(doc_lengths, 99)),
}
summary

{'num_docs': 50000,
 'total_tokens': 62021846,
 'mean': 1240.43692,
 'median': 733.0,
 'min': 74,
 'max': 123565,
 'p90': 2454.0,
 'p95': 3592.0,
 'p99': 8602.180000000037}

In [ ]:
thresholds = [128, 256, 512, 1024, 2048, 4096, 8192]
coverage = {
    f">= {t}": float((doc_lengths >= t).mean())
    for t in thresholds
}
coverage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram clipped at p99 for readability.
clip = np.percentile(doc_lengths, 99)
axes[0].hist(doc_lengths[doc_lengths <= clip], bins=100, color="#2c7fb8", alpha=0.9)
axes[0].set_title(f"Document lengths (<= p99 = {clip:.0f} tokens)")
axes[0].set_xlabel("tokens")
axes[0].set_ylabel("count")

# CCDF on log-log axes to show the tail.
sorted_lengths = np.sort(doc_lengths)
ccdf = 1.0 - np.arange(1, sorted_lengths.size + 1) / sorted_lengths.size
axes[1].plot(sorted_lengths, np.maximum(ccdf, 1e-8), color="#d95f0e")
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_title("Document length CCDF")
axes[1].set_xlabel("tokens")
axes[1].set_ylabel("P(length >= x)")

plt.tight_layout()
plt.show()